In [2]:
try:
    from core.raft_stereo import RAFTStereo
except ImportError:
    import os

    os.chdir("/RAFT-Stereo")
    from core.raft_stereo import RAFTStereo

import torch
from csstereo.dpn import DPN


# 예측 함수
def predict(dpnet, input_left, input_right):
    dpnet.eval()
    print(input_left.shape, input_right.shape)
    
    # 전처리: 0~1로 정규화 후 GPU로 이동
    input_left = torch.clamp(input_left / 255.0, 0.0, 1.0).cuda()
    input_right = torch.clamp(input_right / 255.0, 0.0, 1.0).cuda()
    
    with torch.no_grad():
        # DPN을 사용한 깊이 추정
        ldisps, rdisps = dpnet(input_left, input_right)
        
        return ldisps[-1] * 512

# 모델 초기화
ckpt_path = 'csstereo_pretrained.pth'  # 체크포인트 파일 경로 설정

dpnet = DPN(in_shape=(540,720))

checkpoint = torch.load(ckpt_path)
dpnet.load_state_dict(checkpoint['dpnet'])

dpnet = dpnet.cuda()
#dpnet = nn.DataParallel(dpnet)


# 예시 입력 데이터 생성 (크기에 맞게 입력 필요)
input_left = torch.randn(1, 3, 540, 720)  # 예시 입력 좌측 이미지
input_right = torch.randn(1, 1, 540, 720)  # 예시 입력 우측 이미지

# 예측 수행
ldisps, rdisps = predict(dpnet,input_left, input_right)

print("Left Disparity:", ldisps)
print("Right Disparity:", rdisps)



torch.Size([1, 3, 540, 720]) torch.Size([1, 1, 540, 720])


ValueError: not enough values to unpack (expected 2, got 1)

In [3]:
from matplotlib import pyplot as plt
from myutils.hy5py import get_frame_by_path
from myutils.image_process import read_image_pair
from myutils.widget import FrameExplorer


def plot_csstereo(frame_path: str):
    images = read_image_pair(frame_path)
    with get_frame_by_path(frame_path) as f:
        disparity = f["disparity/bpnet"][:] if "disparity/bpnet" in f else f["disparity/rgb"][:]
    image_left = torch.from_numpy(images[0]).permute(2,0,1).unsqueeze(0).float()#[...,128:128+256, 128:128+512]
    image_right = torch.from_numpy(images[3]).unsqueeze(0).unsqueeze(0).float()#[...,128:128+256, 128:128+512]
    ldis, rdis = predict(dpnet, image_left, image_right)
    print(ldis[-1][0,0].shape)
    plt.subplot(1,3,1)
    plt.imshow(ldis[0][0,0].cpu()*1024,cmap="magma", vmin=0, vmax=32 )
    plt.colorbar()
    plt.subplot(1,3,2)
    plt.imshow(ldis[-1][0,0].cpu()*1024,cmap="magma", vmin=0, vmax=32 )
    plt.colorbar()
    plt.subplot(1,3,3)
    plt.imshow(disparity,cmap="magma", vmin=0, vmax=32)
    plt.colorbar()
    plt.show()

FrameExplorer(plot_csstereo)
    
    

In [ ]:
import time
import cv2
import numpy as np
import pfmread
def predict_disparity(image_left: torch.Tensor, image_right: torch.Tensor):
    cnt = 0
    
    left = image_left[0].permute(1,2,0).cpu().numpy().astype(np.uint8)
    right = image_right[0].permute(1,2,0).cpu().numpy().astype(np.uint8)
    cv2.imwrite("CREStereo/left.png", left)
    cv2.imwrite("CREStereo/right.png", right)
    while True:
        cnt += 1
        time.sleep(1)
        if os.path.exists("CREStereo/output.png"):
            disp = pfmread.read("CREStereo/output.png")[...,0]
            os.remove("CREStereo/output.png")
            return torch.from_numpy(disp)
        if cnt > 10:
            raise Exception
        
    

    

In [23]:
def plot_csstereo(frame_path: str):
    images = read_image_pair(frame_path)
    with get_frame_by_path(frame_path) as f:
        disparity = f["disparity/bpnet"][:] if "disparity/bpnet" in f else f["disparity/rgb"][:]
    image_left = torch.from_numpy(images[0]).permute(2,0,1).unsqueeze(0).float()#[...,128:128+256, 128:128+512]
    image_right = torch.from_numpy(images[1]).permute(2,0,1).unsqueeze(0).float()#[...,128:128+256, 128:128+512]
    ldis = predict_disparity(image_left, image_right)
    print(ldis.shape)
    plt.subplot(1,3,1)
    plt.imshow(ldis,cmap="magma", vmin=0, vmax=32 )

    plt.subplot(1,3,3)
    plt.imshow(disparity,cmap="magma", vmin=0, vmax=32)
    plt.colorbar()
    plt.show()

FrameExplorer(plot_csstereo)